# Transforme sua AWS Lambda em MCP com Gateway OInbound Auth
## Transforme funções AWS Lambda em ferramentas MCP seguras com o Bedrock AgentCore Gateway

## Visão Geral
O Bedrock AgentCore Gateway oferece aos clientes uma forma de transformar suas funções AWS Lambda existentes em servidores MCP totalmente gerenciados, sem a necessidade de gerenciar infraestrutura ou hospedagem. O Gateway fornece uma interface uniforme do Model Context Protocol (MCP) para todas essas ferramentas. O Gateway emprega um modelo de autenticação dupla para garantir o controle de acesso seguro tanto para inbound requests quanto para outbound connections aos recursos de destino. O framework consiste em dois componentes principais: Inbound Auth (Inbound Auth), que valida e autoriza usuários que tentam acessar os alvos do gateway, e Outbound Auth (Outbound Auth), que permite ao gateway conectar-se com segurança aos recursos de backend em nome dos usuários autenticados. O Gateway utiliza uma role IAM para autorizar as chamadas às funções AWS Lambda para outbound auth.

Neste exemplo, demonstraremos OAuth para inbound auth e roles IAM para outbound auth.

![Como funciona](images/lambda-iam-gateway.png)

### Detalhes do Tutorial


| Informação               | Detalhes                                                      |
|:-------------------------|:--------------------------------------------------------------|
| Tipo do tutorial         | Interativo                                                    |
| Componentes AgentCore    | AgentCore Gateway, AgentCore Identity                         |
| Framework de Agentes     | Strands Agents                                                |
| Tipo de alvo do Gateway  | AWS Lambda                                                    |
| Inbound Auth IdP   | Amazon Cognito                                                |
| Outbound Auth            | AWS IAM                                                       |
| Modelo LLM              | Anthropic Claude Haiku 4.5, Amazon Nova Pro                   |
| Componentes do tutorial  | Criação e Invocação do AgentCore Gateway                      |
| Vertical do tutorial     | Cross-vertical                                                |
| Complexidade do exemplo  | Fácil                                                         |
| SDK utilizado            | boto3                                                         |

Na primeira parte do tutorial, criaremos alguns alvos (targets) do AgentCore Gateway

### Arquitetura do Tutorial
Neste tutorial, transformaremos operações definidas em funções AWS Lambda em ferramentas MCP e as hospedaremos no Bedrock AgentCore Gateway.
Para fins de demonstração, utilizaremos um Agente Strands usando modelos Amazon Bedrock.
No nosso exemplo, usaremos um agente muito simples com duas ferramentas: get_order e update_order.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

## Configurando Autenticação para Requisições de Entrada do AgentCore Gateway
O AgentCore Gateway fornece conexões seguras via inbound auth e saída. Para a inbound auth, o AgentCore Gateway analisa o token OAuth passado durante a invocação para decidir permitir ou negar o acesso a uma ferramenta no gateway. Se uma ferramenta precisa de acesso a recursos externos, o AgentCore Gateway pode usar outbound auth via API Key, IAM ou Token OAuth para permitir ou negar o acesso ao recurso externo.



Durante o fluxo de inbound auth, um agente ou o cliente MCP chama uma ferramenta MCP no AgentCore Gateway adicionando um token de acesso OAuth (gerado pelo IdP do usuário). O AgentCore Gateway então valida o token de acesso OAuth e realiza a inbound auth.

Se a ferramenta executando no AgentCore Gateway precisa acessar recursos externos, o OAuth recuperará as credenciais dos recursos downstream usando o provedor de credenciais de recurso para o alvo do Gateway. O AgentCore Gateway passa as credenciais de autorização ao chamador para obter acesso à API downstream. 

In [27]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [28]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1') # set the AWS region

In [29]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [30]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda function created with ARN: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda function creation failed with message: ", lambda_resp['lambda_function_arn'])

Reading code from zip file
Creating IAM role for lambda function
Attaching policy to the IAM role
Role 'gateway_lambda_iamrole' created successfully: arn:aws:iam::134266074494:role/gateway_lambda_iamrole
Creating lambda function
Lambda function created with ARN:  arn:aws:lambda:us-east-1:134266074494:function:gateway_lambda


In [31]:
#### Create an IAM role for the Gateway to assume
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

attaching role policy agentcore-sample-lambdagateway-role
Agentcore gateway role ARN:  arn:aws:iam::134266074494:role/agentcore-sample-lambdagateway-role


# Criar Pool do Amazon Cognito para inbound auth no Gateway

In [32]:
# Creating Cognito User Pool 
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# Get discovery URL  
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

Creating or retrieving Cognito resources...
Creating new user pool
Domain created as well
User Pool ID: us-east-1_1VNAVAvZX
creating new resource server
Resource server ensured.
creating new m2m client
Client ID: 4u96u0f8r9lvbrisdsv090pgb4
https://cognito-idp.us-east-1.amazonaws.com/us-east-1_1VNAVAvZX/.well-known/openid-configuration


# Criar o Gateway com Autorizador Amazon Cognito para inbound auth

In [33]:
# CreateGateway with Cognito authorizer without CMK. Use the Cognito user pool created in the previous step
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='TestGWforLambda',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway with AWS Lambda target type'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

{'ResponseMetadata': {'RequestId': '63237350-cda5-4b61-95f3-0f60cd66fd76', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Mon, 23 Mar 2026 11:47:19 GMT', 'content-type': 'application/json', 'content-length': '815', 'connection': 'keep-alive', 'x-amzn-requestid': '63237350-cda5-4b61-95f3-0f60cd66fd76', 'x-amzn-remapped-x-amzn-requestid': '63237350-cda5-4b61-95f3-0f60cd66fd76', 'x-amzn-remapped-content-length': '815', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'arM7RHaNoAMEG5g=', 'x-amzn-trace-id': 'Root=1-69c12847-2a8007b94396ad4317381e30', 'x-amzn-remapped-date': 'Mon, 23 Mar 2026 11:47:19 GMT'}, 'RetryAttempts': 0}, 'gatewayArn': 'arn:aws:bedrock-agentcore:us-east-1:134266074494:gateway/testgwforlambda-qbyiwa9j9l', 'gatewayId': 'testgwforlambda-qbyiwa9j9l', 'gatewayUrl': 'https://testgwforlambda-qbyiwa9j9l.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp', 'createdAt': datetime.datetime(2026, 3, 23, 11, 47, 19, 870258, tzinfo=tzutc()), 'updatedAt': datetime.da

# Criar um alvo AWS Lambda e transformar em ferramentas MCP

## Recuperar a ARN da função Lambda
A célula abaixo busca a ARN da função Lambda criada anteriormente e armazena na variável `lambda_function_arn` para uso na configuração do target.

In [34]:
import boto3

LAMBDA_FUNCTION_NAME = 'gateway_lambda'  # Nome da função Lambda criada anteriormente

lambda_client = boto3.client('lambda', region_name=os.environ['AWS_DEFAULT_REGION'])

try:
    response = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
    lambda_function_arn = response['Configuration']['FunctionArn']
    print(f"Lambda ARN encontrada: {lambda_function_arn}")
except Exception as e:
    print(f"Erro ao buscar a Lambda: {e}")
    lambda_function_arn = None

Lambda ARN encontrada: arn:aws:lambda:us-east-1:134266074494:function:gateway_lambda


In [35]:
# Configuração do target Lambda usando a ARN recuperada na célula anterior
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_function_arn, # ARN da Lambda recuperada automaticamente
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Lambda Target using SDK',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Chamando o Bedrock AgentCore Gateway a partir de um Agente Strands

O agente Strands integra-se perfeitamente com ferramentas AWS através do Bedrock AgentCore Gateway, que implementa a especificação do Model Context Protocol (MCP). Esta integração permite comunicação segura e padronizada entre agentes de IA e serviços AWS.

Em sua essência, o Bedrock AgentCore Gateway serve como um Gateway compatível com o protocolo que expõe APIs MCP fundamentais: ListTools e InvokeTools. Essas APIs permitem que qualquer cliente ou SDK compatível com MCP descubra e interaja com ferramentas disponíveis de forma segura e padronizada. Quando o agente Strands precisa acessar serviços AWS, ele se comunica com o Gateway usando esses endpoints padronizados pelo MCP.

A implementação do Gateway adere estritamente à (especificação de Autorização MCP)[https://modelcontextprotocol.org/specification/draft/basic/authorization], garantindo segurança robusta e controle de acesso. Isso significa que cada invocação de ferramenta pelo agente Strands passa por uma etapa de autorização, mantendo a segurança enquanto habilita funcionalidades poderosas.

Por exemplo, quando o agente Strands precisa acessar ferramentas MCP, ele primeiro chama ListTools para descobrir as ferramentas disponíveis e depois usa InvokeTools para executar ações específicas. O Gateway lida com todas as validações de segurança necessárias, traduções de protocolo e interações com serviços, tornando todo o processo transparente e seguro.

Essa abordagem arquitetural significa que qualquer cliente ou SDK que implemente a especificação MCP pode interagir com serviços AWS através do Gateway, tornando-o uma solução versátil e preparada para o futuro para integrações de agentes de IA.

![Agente Strands chamando o Gateway](images/strands-lambda-gateway.png)

# Solicitar o token de acesso do Amazon Cognito para inbound auth

In [36]:
import time
time.sleep(10)

In [37]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("Token response:", token)

Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes
4u96u0f8r9lvbrisdsv090pgb4
Token response: eyJraWQiOiJMNkpkdnRzbzRFT3lDbWxVeXhueUF0QnhjbnZ3ZjRUZ0tJZWk4VkdvTFRrPSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiI0dTk2dTBmOHI5bHZicmlzZHN2MDkwcGdiNCIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoic2FtcGxlLWFnZW50Y29yZS1nYXRld2F5LWlkXC9nYXRld2F5OndyaXRlIHNhbXBsZS1hZ2VudGNvcmUtZ2F0ZXdheS1pZFwvZ2F0ZXdheTpyZWFkIiwiYXV0aF90aW1lIjoxNzc0MjY2NDY5LCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAudXMtZWFzdC0xLmFtYXpvbmF3cy5jb21cL3VzLWVhc3QtMV8xVk5BVkF2WlgiLCJleHAiOjE3NzQyNzAwNjksImlhdCI6MTc3NDI2NjQ2OSwidmVyc2lvbiI6MiwianRpIjoiZmU4ZjcyYTQtMzk3Mi00ZWU2LTkwYTQtMzQ1MzIyOGI1YzM3IiwiY2xpZW50X2lkIjoiNHU5NnUwZjhyOWx2YnJpc2RzdjA5MHBnYjQifQ.Lb0AX_DecdbhZSbWaP2AQE6Jo_qLl6hIReqw8p2KEU1H8DdEa90L5YGmTZ5Hr8DX7aT9gjATev8SzWr1VXpFwRokxJ-jQQZE7fDkn8R7AJMMEr_vjh-pE90pbIyViKG_DKb21WVV9i10UNN2SFn5VMCdgE6vfxrtonGAa1g62CLfgHtoklGEl1DWESmR_DZiuGrbH5RlMEejIZI384fCm4MykPcUOJJiXd8

# Agente Strands chamando ferramentas MCP do AWS Lambda usando o Bedrock AgentCore Gateway

In [38]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL,headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

## The IAM credentials configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [39]:
from strands import Agent
import logging


# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

with client:
    # Call the listTools 
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(model=yourmodel,tools=tools) ## you can replace with any model you like
    print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")
    # Invoke the agent with the sample prompt. This will only invoke  MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi , can you list all tools available to you")
    # Invoke the agent with sample prompt, invoke the tool and display the response
    agent("Check the order status for order id 123 and show me the exact response from the tool")
    # Call the MCP tool explicitly. The MCP Tool name and arguments must match with your AWS Lambda function or the OpenAPI/Smithy API
    result = client.call_tool_sync(
    tool_use_id="get-order-id-123-call-1", # You can replace this with unique identifier. 
    name=targetname+"___get_order_tool", # This is the tool name based on AWS Lambda target types. This will change based on the target name
    arguments={"orderId": "123"}
    )
    # Print the MCP Tool response
    print(f"Tool Call result: {result['content'][0]['text']}")


Tools loaded in the agent are ['LambdaUsingSDK___get_order_tool', 'LambdaUsingSDK___update_order_tool']
<thinking> I need to list all the tools that are available to me. </thinking>

The tools available to me are:

1. **LambdaUsingSDK___get_order_tool**
   - Description: Tool to get the order.
   - Parameters:
     - `orderId` (required): The ID of the order.

2. **LambdaUsingSDK___update_order_tool**
   - Description: Tool to update the order.
   - Parameters:
     - `orderId` (required): The ID of the order.<thinking> To check the order status for order ID 123, I need to use the `LambdaUsingSDK___get_order_tool` with the provided `orderId`. </thinking>

Tool #1: LambdaUsingSDK___get_order_tool
<thinking> The tool has returned the status of the order with ID 123. I will now present the exact response from the tool to the user. </thinking>

The exact response from the tool is:
```
{
  "statusCode": 200,
  "body": "Order Id 123 is in shipped status"
}
```

This indicates that the order 

**Problema: se você receber o erro abaixo ao executar a célula abaixo, isso indica incompatibilidade entre as versões do pydantic e pydantic-core.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**Como resolver?**

Você precisará garantir que tenha pydantic==2.7.2 e pydantic-core 2.27.2, que são compatíveis entre si. Reinicie o kernel após a instalação.

# Limpeza

A célula abaixo deleta **todos** os recursos criados neste tutorial:
1. AgentCore Gateway (e seus targets)
2. Função AWS Lambda (`gateway_lambda`)
3. IAM Role da Lambda (`gateway_lambda_iamrole`)
4. IAM Role do Gateway (`agentcore-sample-lambdagateway-role`)
5. Cognito User Pool (`sample-agentcore-gateway-pool`)

## Excluir todos os recursos criados (Opcional)

In [40]:
import boto3
import time
import utils
from botocore.exceptions import ClientError

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

# --- 1. Deletar o AgentCore Gateway (e seus targets) ---
print("=" * 60)
print("1. Deletando AgentCore Gateway e targets...")
print("=" * 60)
try:
    utils.delete_gateway(gateway_client, gatewayID)
    print(f"✅ Gateway {gatewayID} deletado com sucesso.")
except Exception as e:
    print(f"⚠️ Erro ao deletar gateway: {e}")

# --- 2. Deletar a função Lambda ---
print("\n" + "=" * 60)
print("2. Deletando função Lambda...")
print("=" * 60)
lambda_client = boto3.client('lambda', region_name=REGION)
try:
    lambda_client.delete_function(FunctionName='gateway_lambda')
    print("✅ Lambda 'gateway_lambda' deletada com sucesso.")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print("⚠️ Lambda 'gateway_lambda' já não existe.")
    else:
        print(f"⚠️ Erro ao deletar Lambda: {e}")

# --- 3. Deletar IAM Role da Lambda ---
print("\n" + "=" * 60)
print("3. Deletando IAM Role da Lambda...")
print("=" * 60)
iam_client = boto3.client('iam')
lambda_role_name = 'gateway_lambda_iamrole'
try:
    # Desanexar managed policies
    attached = iam_client.list_attached_role_policies(RoleName=lambda_role_name)
    for policy in attached['AttachedPolicies']:
        iam_client.detach_role_policy(RoleName=lambda_role_name, PolicyArn=policy['PolicyArn'])
        print(f"   Desanexada policy: {policy['PolicyArn']}")
    # Deletar inline policies
    inline = iam_client.list_role_policies(RoleName=lambda_role_name)
    for policy_name in inline['PolicyNames']:
        iam_client.delete_role_policy(RoleName=lambda_role_name, PolicyName=policy_name)
        print(f"   Deletada inline policy: {policy_name}")
    # Deletar role
    iam_client.delete_role(RoleName=lambda_role_name)
    print(f"✅ IAM Role '{lambda_role_name}' deletada com sucesso.")
except ClientError as e:
    if e.response['Error']['Code'] == 'NoSuchEntity':
        print(f"⚠️ IAM Role '{lambda_role_name}' já não existe.")
    else:
        print(f"⚠️ Erro ao deletar IAM Role: {e}")

# --- 4. Deletar IAM Role do Gateway ---
print("\n" + "=" * 60)
print("4. Deletando IAM Role do Gateway...")
print("=" * 60)
gateway_role_name = 'agentcore-sample-lambdagateway-role'
try:
    # Desanexar managed policies
    attached = iam_client.list_attached_role_policies(RoleName=gateway_role_name)
    for policy in attached['AttachedPolicies']:
        iam_client.detach_role_policy(RoleName=gateway_role_name, PolicyArn=policy['PolicyArn'])
        print(f"   Desanexada policy: {policy['PolicyArn']}")
    # Deletar inline policies
    inline = iam_client.list_role_policies(RoleName=gateway_role_name)
    for policy_name in inline['PolicyNames']:
        iam_client.delete_role_policy(RoleName=gateway_role_name, PolicyName=policy_name)
        print(f"   Deletada inline policy: {policy_name}")
    # Deletar role
    iam_client.delete_role(RoleName=gateway_role_name)
    print(f"✅ IAM Role '{gateway_role_name}' deletada com sucesso.")
except ClientError as e:
    if e.response['Error']['Code'] == 'NoSuchEntity':
        print(f"⚠️ IAM Role '{gateway_role_name}' já não existe.")
    else:
        print(f"⚠️ Erro ao deletar IAM Role: {e}")

# --- 5. Deletar Cognito User Pool ---
print("\n" + "=" * 60)
print("5. Deletando Cognito User Pool...")
print("=" * 60)
cognito_client = boto3.client('cognito-idp', region_name=REGION)
try:
    # Primeiro precisa deletar o domain do user pool
    response = cognito_client.describe_user_pool(UserPoolId=user_pool_id)
    domain = response.get('UserPool', {}).get('Domain')
    if domain:
        cognito_client.delete_user_pool_domain(Domain=domain, UserPoolId=user_pool_id)
        print(f"   Deletado domain: {domain}")
        time.sleep(3)
    # Deletar o user pool
    cognito_client.delete_user_pool(UserPoolId=user_pool_id)
    print(f"✅ Cognito User Pool '{user_pool_id}' deletado com sucesso.")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f"⚠️ Cognito User Pool '{user_pool_id}' já não existe.")
    else:
        print(f"⚠️ Erro ao deletar Cognito User Pool: {e}")

print("\n" + "=" * 60)
print("🧹 Limpeza concluída!")
print("=" * 60)

1. Deletando AgentCore Gateway e targets...
Deleting all targets for gateway testgwforlambda-qbyiwa9j9l
Deleting target  HNJE97NSIW
Deleting gateway  testgwforlambda-qbyiwa9j9l
✅ Gateway testgwforlambda-qbyiwa9j9l deletado com sucesso.

2. Deletando função Lambda...
✅ Lambda 'gateway_lambda' deletada com sucesso.

3. Deletando IAM Role da Lambda...
   Desanexada policy: arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole
✅ IAM Role 'gateway_lambda_iamrole' deletada com sucesso.

4. Deletando IAM Role do Gateway...
   Deletada inline policy: AgentCorePolicy
✅ IAM Role 'agentcore-sample-lambdagateway-role' deletada com sucesso.

5. Deletando Cognito User Pool...
   Deletado domain: us-east-11vnavavzx
✅ Cognito User Pool 'us-east-1_1VNAVAvZX' deletado com sucesso.

🧹 Limpeza concluída!
